# Extraction d'embeddings avec DaSheng

Ce notebook extrait des embeddings audio avec le modèle DaSheng.

## Étapes :
1. Monte Google Drive si nécessaire
2. Installe les dépendances
3. Charge un fichier ZIP contenant des fichiers audio (.wav)
4. Rééchantillonne chaque audio vers 16 kHz
5. Calcule les embeddings avec le modèle pré-entraîné DaSheng
6. Sauvegarde les embeddings dans un fichier .npy et les métadonnées dans un fichier .csv

## Cell 1: Google Colab Setup & Google Drive Mount

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully')
else:
    print('Running locally (not in Google Colab)')

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    PROJECT_PATH = 'C:/Users/moutier/Desktop/CNRS/CNRS'

print(f'Project path: {PROJECT_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully
Project path: /content/drive/MyDrive/audio_simon_moutier


## Cell 2: Install Dependencies

In [2]:
if IN_COLAB:
    print('Installing required packages')
    !pip install -q dasheng torch torchaudio numpy pandas tqdm
    print('All packages installed')

Installing required packages
All packages installed


## Cell 3: Imports

In [3]:
import os
import zipfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from dasheng import dasheng_base
from tqdm import tqdm

print('All imports successful')

All imports successful


## Cell 4: Configuration

In [4]:
# Paths
ZIP_PATH = os.path.join(PROJECT_PATH, 'Data/Audio/AudioRaw/3s_samples_subset.zip')

OUTPUT_DIR = os.path.join(PROJECT_PATH, 'embeddings')
OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, 'dasheng_embeddings.parquet')

# Settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16000

print(f"Device: {DEVICE}")
print(f"ZIP path: {ZIP_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Output parquet file: {OUTPUT_PARQUET}")

# Verify paths
if not os.path.exists(ZIP_PATH):
    print(f"WARNING: ZIP file not found at {ZIP_PATH}")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

Device: cuda
ZIP path: /content/drive/MyDrive/audio_simon_moutier/Data/Audio/AudioRaw/3s_samples_subset.zip
Output directory: /content/drive/MyDrive/audio_simon_moutier/embeddings
Output parquet file: /content/drive/MyDrive/audio_simon_moutier/embeddings/dasheng_embeddings.parquet


In [5]:
# ============================================================
# DOWNLOAD DASHENG CHECKPOINT MANUALLY
# ============================================================

import os
import urllib.request

CHECKPOINT_DIR = "/content/drive/MyDrive/audio_simon_moutier/Scripts/Embedding_Extraction/embed_extraction_dasheng"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "dasheng_base.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(CHECKPOINT_PATH):

    print("Downloading DaSheng checkpoint manually...")

    url = "https://zenodo.org/records/11511780/files/dasheng_base.pt?download=1"

    urllib.request.urlretrieve(url, CHECKPOINT_PATH)

    print("Download completed")

else:
    print("Checkpoint already exists")

Checkpoint already exists


## Cell 5: Load DaSheng Model

In [6]:
print('Loading DaSheng model')
print(f'Using device: {DEVICE}')

model = dasheng_base(path=CHECKPOINT_PATH).to(DEVICE)
model.eval()

print('Model loaded successfully')

Loading DaSheng model
Using device: cuda
Model loaded successfully


## Cell 6: Define embedding Extraction Function

In [7]:
@torch.no_grad()
def extract_embedding(audio_path):
    """
    Extract DaSheng embedding from an audio file.

    Parameters:
    -----------
    audio_path : str
        Path to the audio file (.wav)

    Returns:
    --------
    embedding : np.ndarray
        1D embedding vector.
    """

    waveform, sr = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sr,
            new_freq=TARGET_SR,
        )

    waveform = waveform.float().to(DEVICE)

    features = model(waveform)
    embedding = features.mean(dim=1)

    return embedding.squeeze(0).cpu().numpy()


print('Embedding extraction function defined')

Embedding extraction function defined


## Cell 7: Extract Embeddings from ZIP

In [8]:
all_embeddings = []
all_filenames = []
error_count = 0

with tempfile.TemporaryDirectory() as tmpdir:

    print('Extracting ZIP file...')

    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(tmpdir)

    audio_files = list(Path(tmpdir).rglob('*.wav'))

    print(f'Found {len(audio_files)} audio files')

    for audio_path in tqdm(audio_files, desc='Extracting embeddings'):
        try:
            emb = extract_embedding(str(audio_path))
            all_embeddings.append(emb)
            all_filenames.append(audio_path.name)
        except Exception as e:
            error_count += 1
            if error_count <= 5:
                print(f'\nError with {audio_path.name}: {str(e)[:100]}')

print(f"\n{'=' * 60}")
print(f'Successfully extracted: {len(all_embeddings)} files')
print(f'Failed: {error_count} files')
print(f'Success rate: {len(all_embeddings) / len(audio_files) * 100:.1f}%')
print(f"{'=' * 60}")

Extracting ZIP file...
Found 10325 audio files


Extracting embeddings: 100%|██████████| 10325/10325 [02:59<00:00, 57.57it/s]



Successfully extracted: 10325 files
Failed: 0 files
Success rate: 100.0%


## Cell 8 : Save Embeddings

In [9]:
all_embeddings = np.stack(all_embeddings)

# Create a DataFrame with embeddings and filenames
# Each embedding becomes columns
embedding_df = pd.DataFrame(
    all_embeddings,
    columns=[f"embedding_{i}" for i in range(all_embeddings.shape[1])]
)

# Add filenames column at the beginning
embedding_df.insert(0, 'filename', all_filenames)

# Save to Parquet
embedding_df.to_parquet(OUTPUT_PARQUET, index=False)

print("Embeddings saved successfully.")
print(f"Embeddings shape: {all_embeddings.shape}")
print(f"DataFrame shape: {embedding_df.shape}")
print(f"Saved to: {OUTPUT_PARQUET}")

Embeddings saved successfully.
Embeddings shape: (10325, 768)
DataFrame shape: (10325, 769)
Saved to: /content/drive/MyDrive/audio_simon_moutier/embeddings/dasheng_embeddings.parquet


## Cell 9: Summary Statistics

In [10]:
print('\n' + '='*60)
print('EXTRACTION COMPLETE')
print('='*60)
print(f'Total audio files processed: {len(all_embeddings)}')
print(f'Embedding dimension: {all_embeddings.shape[1]}')
print(f'Device used: {DEVICE}')
print(f'Target sample rate: {TARGET_SR} Hz')
print()
print('Output files:')
print(f'  - {OUTPUT_PARQUET}')
print('='*60)


EXTRACTION COMPLETE
Total audio files processed: 10325
Embedding dimension: 768
Device used: cuda
Target sample rate: 16000 Hz

Output files:
  - /content/drive/MyDrive/audio_simon_moutier/embeddings/dasheng_embeddings.parquet
